# Task 2: Retrieval and Generation
### INFO-H-515 Big Data: Distributed Data Management and Scalable Analytics
**Environment:** Google Colab  |  **Branch:** `retrieval-generation`

This notebook implements the full **Retrieval-Augmented Generation** pipeline:
1. Load chunked + embedded data produced by Task 1 (Parquet, PySpark).
2. Embed the user query with the **same strategy** as the corpus (TF-IDF / Word2Vec / SBERT).
3. Retrieve the Top-N most relevant chunks using **PySpark RDD** distributed similarity.
4. Build a grounded, citation-aware prompt and call an LLM to generate exam questions.

In [1]:
# 1. Environment setup
!apt-get install -y -qq default-jre > /dev/null 2>&1
!java -version


openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [2]:
!pip install -q \
    pyspark \
    sentence-transformers \
    transformers \
    gensim \
    scikit-learn \
    pyarrow \
    bitsandbytes \
    accelerate \
    requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Configuration

| Parameter | Description |
|---|---|
| `QUERY` | Natural-language query for retrieval |
| `EMBEDDING_STRATEGY` | `tfidf` / `word2vec` / `sbert` - must match Task 1 |
| `SIMILARITY_STRATEGY` | `cosine` / `euclidean` |
| `NUMBER_OF_CHUNKS_TO_RETRIEVE` | Top_N chunks to retrieve |
| `NUMBER_OF_QUESTIONS_TO_GENERATE` | Number of exam questions to generate |
| `GENERATOR_MODEL` | HuggingFace model ID (local, 4-bit) |
| `MAX_CONTEXT_TOKENS` | Maximum tokens for the LLM context window |
| `INPUT_DIR` | Path to Task 1 Parquet output on Google Drive |
| `OUTPUT_DIR` | Where to save results |

In [ ]:
from google.colab import userdata

# Query
QUERY = "Generate exam questions on Big Data frameworks and distributed computing with Spark"

# Retrieval
EMBEDDING_STRATEGY              = "sbert"    # "tfidf" | "word2vec" | "sbert"
SIMILARITY_STRATEGY             = "cosine"   # "cosine" | "euclidean"
NUMBER_OF_CHUNKS_TO_RETRIEVE    = 5
NUMBER_OF_QUESTIONS_TO_GENERATE = 3

# Generation
GENERATOR_MODEL    = "meta-llama/Llama-3.2-3B-Instruct"
# GENERATOR_MODEL  = "mistralai/Mistral-7B-Instruct-v0.3"   # slower but stronger
MAX_CONTEXT_TOKENS = 3000
HF_TOKEN           = userdata.get("HF_TOKEN")

# Paths
INPUT_DIR  = "/content/drive/MyDrive/DDM/distributed-retrieval-augmented-generation-team-18/processed_data"
OUTPUT_DIR = "/content/drive/MyDrive/DDM/distributed-retrieval-augmented-generation-team-18/task2_output"

# Spark
PARALLELISM = 4

# Embedding hyper-params (must match Task 1 configuration)
SBERT_MODEL    = "all-MiniLM-L6-v2"
WORD2VEC_DIM   = 128
TFIDF_MAX_FEAT = 10_000
TFIDF_SVD_DIM  = 256

## 1. Spark Session

In [5]:
import os, json, time, re, subprocess, glob
from typing import List, Dict, Tuple, Optional

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, FloatType


def _find_java_home() -> str:
    """Detect JAVA_HOME dynamically - works across different Colab images."""
    # 1. Already set in environment
    if os.environ.get("JAVA_HOME"):
        return os.environ["JAVA_HOME"]
    # 2. Derive from `which java`
    try:
        java_bin = subprocess.check_output(["which", "java"], text=True).strip()
        if java_bin:
            # resolve symlinks: /usr/bin/java → .../jre/bin/java
            real = subprocess.check_output(["readlink", "-f", java_bin], text=True).strip()
            # go up two levels: bin/java → jvm root
            return os.path.dirname(os.path.dirname(real))
    except Exception:
        pass
    # 3. Scan /usr/lib/jvm for the first available JDK/JRE
    candidates = glob.glob("/usr/lib/jvm/java-*") + glob.glob("/usr/lib/jvm/default*")
    if candidates:
        return candidates[0]
    raise RuntimeError("Java not found. Run: !apt-get install -y default-jre")


def build_spark(parallelism: int = PARALLELISM) -> SparkSession:
    java_home = _find_java_home()
    os.environ["JAVA_HOME"] = java_home
    print(f"JAVA_HOME set to: {java_home}")
    return (
        SparkSession.builder
        .appName("INFO-H515 Task2 Retrieval & Generation")
        .master(f"local[{parallelism}]")
        .config("spark.default.parallelism",    str(parallelism))
        .config("spark.sql.shuffle.partitions", str(parallelism))
        .config("spark.driver.memory",          "4g")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = build_spark(PARALLELISM)
print(f"Spark version : {spark.version}")
print(f"Parallelism   : {spark.sparkContext.defaultParallelism}")

JAVA_HOME set to: /usr/lib/jvm/java-17-openjdk-amd64
Spark version : 4.0.2
Parallelism   : 4


## 2. Data Loading
Load the Parquet files produced by Task 1 into a **PySpark RDD**.
The `embedding` column (stored as a Spark array) is converted to a Python `list`
so that it can be used in RDD `map` operations on the executors.

In [6]:
def load_chunk_rdd(spark: SparkSession, input_dir: str, embedding_strategy: str):
    """
    Read Task 1 Parquet output for the given embedding strategy.
    Returns a cached RDD of dicts with 'embedding' as a Python list.
    """
    parquet_path = os.path.join(input_dir, f"strategy={embedding_strategy}")
    df = spark.read.parquet(parquet_path)

    print("Schema:")
    df.printSchema()
    print(f"Total chunks loaded: {df.count()}")
    df.select("source_pdf", "page_num", "chunk_id", "embedding_strategy").show(5, truncate=False)

    def row_to_dict(row):
        d = row.asDict()
        d["embedding"] = list(d["embedding"]) if d["embedding"] is not None else []
        return d

    rdd = df.rdd.map(row_to_dict)
    rdd.cache()
    return rdd


chunk_rdd = load_chunk_rdd(spark, INPUT_DIR, EMBEDDING_STRATEGY)
print(f"RDD partitions : {chunk_rdd.getNumPartitions()}")

Schema:
root
 |-- id: string (nullable = true)
 |-- source_pdf: string (nullable = true)
 |-- page_num: integer (nullable = true)
 |-- chunk_id: string (nullable = true)
 |-- start_word: integer (nullable = true)
 |-- end_word: integer (nullable = true)
 |-- chunk_text: string (nullable = true)
 |-- chunk_word_count: integer (nullable = true)
 |-- embedding: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- embedding_strategy: string (nullable = true)
 |-- embedding_dim: integer (nullable = true)

Total chunks loaded: 1872
+-----------------+--------+---------------------------+------------------+
|source_pdf       |page_num|chunk_id                   |embedding_strategy|
+-----------------+--------+---------------------------+------------------+
|6-parallel.pdf   |39      |6-parallel.pdf_p039_c000   |sbert             |
|3-stream_proc.pdf|20      |3-stream_proc.pdf_p020_c000|sbert             |
|5-nosql.pdf      |22      |5-nosql.pdf_p022_c000      |sbert    

In [ ]:
# chunk_rdd.take(1)[0]

## 3. Distance Metrics  *(implemented from scratch)*

Both functions use only built-in Python operations — **no** `scipy`,
`sklearn.metrics.pairwise`, or similar external implementations,
as required by the project specification (section 3.2).

| Strategy | Formula | Higher score = more relevant? |
|---|---|---|
| `cosine` | a·b / (‖a‖ × ‖b‖) | yes (score ∈ [−1, 1]) |
| `euclidean` | −√Σ(aᵢ−bᵢ)² | yes (negated distance) |

In [7]:
# Similarity / distance functions (pure Python, no external metric libs)

def cosine_similarity(a: list, b: list) -> float:
    """Cosine similarity: a·b / (||a|| × ||b||). Returns 0 for zero vectors."""
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return dot / (norm_a * norm_b)


def euclidean_distance(a: list, b: list) -> float:
    """Euclidean (L2) distance: sqrt(sum((a_i - b_i)^2))."""
    return sum((x - y) ** 2 for x, y in zip(a, b)) ** 0.5


# Unit tests
_a, _b = [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]
assert abs(cosine_similarity(_a, _a) - 1.0) < 1e-9,  "cosine self-similarity should be 1"
assert abs(cosine_similarity(_a, _b)) < 1e-9,  "cosine orthogonal should be 0"
assert abs(euclidean_distance(_a, _b) - 2.0 ** 0.5) < 1e-9, "euclidean unit vectors"
print("Distance metric unit tests passed.")

Distance metric unit tests passed.


## 4. Query Embedding

The query must be embedded with the **same strategy and model** used to embed corpus chunks.

- **SBERT**: loads the same pre-trained model (`all-MiniLM-L6-v2`) - no re-fitting needed.
- **Word2Vec**: re-trains on corpus texts collected from the Parquet RDD.
- **TF-IDF**: re-fits the TF-IDF + TruncatedSVD pipeline on corpus texts from the Parquet RDD.

> Re-fitting TF-IDF / Word2Vec on the stored corpus ensures the query is projected
> into the exact same vector space as the stored embeddings.

In [13]:
def _get_corpus_texts(chunk_rdd) -> List[str]:
    """Collect all chunk texts to the driver (needed for TF-IDF / Word2Vec re-fitting)."""
    return chunk_rdd.map(lambda c: c["chunk_text"]).collect()


# SBERT
def embed_query_sbert(query: str, model_name: str = SBERT_MODEL) -> List[float]:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    vec = model.encode([query], convert_to_numpy=True)[0]
    return vec.tolist()


# Word2Vec
def embed_query_word2vec(query: str, chunk_rdd, dim: int = WORD2VEC_DIM) -> List[float]:
    from gensim.models import Word2Vec

    def tokenise(text: str) -> List[str]:
        return re.findall(r"[a-z]+", text.lower())

    print("  Re-training Word2Vec on corpus texts ...")
    corpus_sentences = [tokenise(t) for t in _get_corpus_texts(chunk_rdd)]
    model = Word2Vec(
        corpus_sentences, vector_size=dim, window=5,
        min_count=2, workers=4, sg=1, epochs=5, seed=42
    )
    tokens = tokenise(query)
    vecs = [model.wv[t].tolist() for t in tokens if t in model.wv]
    if not vecs:
        return [0.0] * dim
    return [sum(v[i] for v in vecs) / len(vecs) for i in range(dim)]


# TF-IDF + TruncatedSVD (LSA)
def embed_query_tfidf(
    query: str,
    chunk_rdd,
    max_features: int = TFIDF_MAX_FEAT,
    svd_dim: int = TFIDF_SVD_DIM,
) -> List[float]:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD

    print("  Re-fitting TF-IDF + SVD on corpus texts ...")
    texts = _get_corpus_texts(chunk_rdd)
    vectorizer = TfidfVectorizer(max_features=max_features, sublinear_tf=True)
    tfidf_mat = vectorizer.fit_transform(texts)
    svd = TruncatedSVD(n_components=svd_dim, random_state=42)
    svd.fit(tfidf_mat)
    vec = svd.transform(vectorizer.transform([query]))
    return vec[0].tolist()


# Dispatch ──────────────────────────────────────────────────────────────────
_EMBED_QUERY_FN = {
    "sbert":    lambda q, rdd: embed_query_sbert(q),
    "word2vec": lambda q, rdd: embed_query_word2vec(q, rdd),
    "tfidf":    lambda q, rdd: embed_query_tfidf(q, rdd),
}


def embed_query(query: str, strategy: str, chunk_rdd) -> List[float]:
    fn = _EMBED_QUERY_FN.get(strategy)
    if fn is None:
        raise ValueError(
            f"Unknown embedding strategy '{strategy}'. "
            f"Choose from {list(_EMBED_QUERY_FN)}"
        )
    print(f"Embedding query with '{strategy}' ...")
    vec = fn(query, chunk_rdd)
    print(f"  Query vector dim = {len(vec)}")
    return vec

## 5. Distributed Retrieval  *(PySpark RDD)*

The query embedding is **broadcast** to all Spark executors via `SparkContext.broadcast()`.
Each executor computes similarity scores for its own partition of chunks **in parallel**.
The driver then collects and ranks the global Top-N results using `sortBy` + `take`.

```
chunk_rdd  ──map(score_chunk)──►  scored_rdd  ──sortBy(score, desc)──►  top_n
              (parallel, one                       (shuffle across
               score per chunk)                    partitions)
```

In [8]:
def retrieve_top_chunks(
    spark: SparkSession,
    chunk_rdd,
    query_embedding: List[float],
    similarity_strategy: str,
    top_n: int,
) -> List[Tuple[float, Dict]]:
    """
    Compute similarity scores between the query and every stored chunk
    in parallel using PySpark RDD map + sortBy.
    Returns the top-N (score, chunk) pairs ranked by descending score.
    """
    supported = {"cosine", "euclidean"}
    if similarity_strategy not in supported:
        raise ValueError(
            f"Unknown similarity '{similarity_strategy}'. "
            f"Choose from {supported}"
        )

    bc_query = spark.sparkContext.broadcast(query_embedding)
    bc_strategy = spark.sparkContext.broadcast(similarity_strategy)

    # Capture metric functions in the closure so executors can use them
    _cosine = cosine_similarity
    _euclidean = euclidean_distance

    def score_chunk(chunk: Dict) -> Tuple[float, Dict]:
        q = bc_query.value
        strategy = bc_strategy.value
        emb = chunk.get("embedding", [])

        if not emb:
            return (-float("inf"), chunk)

        if strategy == "cosine":
            score = _cosine(q, emb)
        elif strategy == "euclidean":
            score = -_euclidean(q, emb)   # negate: smaller distance = more relevant
        else:
            score = -float("inf")

        return (score, chunk)

    # PySpark RDD: map (parallel scoring) → sortBy (global ranking) → take(top_n)
    scored_rdd = chunk_rdd.map(score_chunk)
    top_chunks = scored_rdd.sortBy(lambda x: x[0], ascending=False).take(top_n)

    bc_query.unpersist()
    bc_strategy.unpersist()

    return top_chunks

## 6. Prompt Construction

A grounded, citation-aware prompt is built from the retrieved chunks.
Each chunk is numbered `[1]`, `[2]`, … so the model can cite its sources.
Chunks are included until `max_context_tokens` is reached
(rough approximation: 1 word ≈ 1.3 sub-word tokens).

In [9]:
def build_prompt(
    query: str,
    top_chunks: List[Tuple[float, Dict]],
    n_questions: int,
    max_tokens: int = MAX_CONTEXT_TOKENS,
) -> str:
    """Build a grounded, citation-aware prompt from the top-N retrieved chunks."""
    max_words = int(max_tokens / 1.3)
    sources_text = ""
    total_words = 0
    used_indices = []

    for idx, (score, chunk) in enumerate(top_chunks, start=1):
        words = chunk.get("chunk_word_count",
                          len(chunk.get("chunk_text", "").split()))
        if total_words + words > max_words:
            break
        label = (
            f"[{idx}] (source: {chunk['source_pdf']}, "
            f"page {chunk['page_num']}, "
            f"chunk {chunk['chunk_id']})"
        )
        sources_text += f"\n{label}\n{chunk['chunk_text']}\n"
        total_words  += words
        used_indices.append(idx)

    cite_hint = ", ".join(f"[{i}]" for i in used_indices)

    example = (
        "Q: <question text>\n"
        "A: <expected answer key>\n"
        "CITATIONS: <e.g. [1], [2]>\n"
        "---"
    )

    return (
        f"TASK: Generate exactly {n_questions} exam question(s) on the following topic:\n"
        f'"{query}"\n\n'
        f"INSTRUCTIONS:\n"
        f"- Use ONLY the SOURCES provided below.\n"
        f"- If the sources are insufficient, say so explicitly.\n"
        f"- Available source numbers: {cite_hint}.\n"
        f"- You MUST output exactly {n_questions} question(s), no more, no less.\n"
        f"- For EACH question, you MUST follow this exact format (including the --- separator):\n\n"
        f"{example}\n\n"
        f"Do NOT group citations at the end. CITATIONS must appear after each individual answer.\n\n"
        f"SOURCES:{sources_text}\n"
        f"BEGIN OUTPUT:\n"
    )


## 7. LLM Integration  *(Llama-3.2-3B-Instruct)*

Uses `meta-llama/Llama-3.2-3B-Instruct` via HuggingFace Transformers pipeline,
the reference model specified by the project (on par with `llama3.2:3b` in Ollama).

> **HF token required** — set `HF_TOKEN` in the config cell
> (the model is gated on HuggingFace).

In [10]:
_llm_pipeline = None  # loaded once, reused across calls

def _get_pipeline():
    global _llm_pipeline
    if _llm_pipeline is None:
        import torch
        from transformers import pipeline
        print(f'Loading {GENERATOR_MODEL} ...')
        _llm_pipeline = pipeline(
            'text-generation',
            model=GENERATOR_MODEL,
            torch_dtype=torch.float16,
            device_map='auto',
            do_sample=False,
            token=HF_TOKEN or None,
        )
        print('Model loaded.')
    return _llm_pipeline


def generate_hf_local(
    prompt:         str,
    model_name:     str = GENERATOR_MODEL,
    max_new_tokens: int = 512,
    **kwargs,
) -> str:
    """Generate with Llama-3.2-3B-Instruct via HuggingFace pipeline."""
    llm = _get_pipeline()
    # Pass messages — pipeline uses Llama's built-in chat template
    messages = [{'role': 'user', 'content': prompt}]
    output   = llm(messages, max_new_tokens=max_new_tokens)
    # output[0]['generated_text'] is a list of message dicts
    return output[0]['generated_text'][-1]['content']


def generate_questions(
    prompt:          str,
    generator_model: str = GENERATOR_MODEL,
    max_new_tokens:  int = 512,
    **kwargs,
) -> str:
    return generate_hf_local(prompt, model_name=generator_model,
                             max_new_tokens=max_new_tokens, **kwargs)

## 8. Full Pipeline

End-to-end execution:

```
load_chunk_rdd
    → embed_query          (same strategy as corpus)
    → retrieve_top_chunks  (PySpark RDD: map + sortBy + take)
    → build_prompt         (grounded, citation-aware)
    → generate_questions   (LLM: HuggingFace local or API)
    → save result.json
```

In [14]:
def run_pipeline(
    query:                        str = QUERY,
    embedding_strategy:           str = EMBEDDING_STRATEGY,
    similarity_strategy:          str = SIMILARITY_STRATEGY,
    number_of_chunks_to_retrieve: int = NUMBER_OF_CHUNKS_TO_RETRIEVE,
    number_of_questions:          int = NUMBER_OF_QUESTIONS_TO_GENERATE,
    generator_model:              str = GENERATOR_MODEL,
    max_context_tokens:           int = MAX_CONTEXT_TOKENS,
    input_dir:                    str = INPUT_DIR,
    output_dir:                   str = OUTPUT_DIR,
) -> Dict:
    """
    Full Task 2 pipeline.

    Returns a dict with:
        query, embedding_strategy, similarity_strategy,
        generation_params, retrieved_chunks (ranked with scores + metadata),
        prompt, generated_output, elapsed_seconds.
    """
    t0 = time.time()

    # 1. Load
    print("[1/5] Loading Parquet data ...")
    rdd = load_chunk_rdd(spark, input_dir, embedding_strategy)

    # 2. Embed query
    print("[2/5] Embedding query ...")
    query_emb = embed_query(query, embedding_strategy, rdd)

    # 3. Retrieve top-N
    print(f"[3/5] Retrieving top-{number_of_chunks_to_retrieve} chunks "
          f"(similarity={similarity_strategy}) ...")
    top_chunks = retrieve_top_chunks(
        spark, rdd, query_emb, similarity_strategy, number_of_chunks_to_retrieve
    )

    # 4. Build prompt
    print("[4/5] Building grounded prompt ...")
    prompt = build_prompt(query, top_chunks, number_of_questions, max_context_tokens)

    # 5. Generate
    print(f"[5/5] Calling {generator_model} ...")
    t_gen  = time.time()
    output = generate_questions(prompt, generator_model=generator_model)
    gen_latency = round(time.time() - t_gen, 2)

    elapsed = round(time.time() - t0, 2)

    result = {
        "query":               query,
        "embedding_strategy":  embedding_strategy,
        "similarity_strategy": similarity_strategy,
        "generation_params": {
            "generator_model":            generator_model,
            "max_context_tokens":         max_context_tokens,
            "number_of_questions":        number_of_questions,
            "number_of_chunks_retrieved": len(top_chunks),
            "generation_latency_s":       gen_latency,
        },
        "retrieved_chunks": [
            {
                "rank":       i + 1,
                "score":      round(score, 6),
                "id":         ch.get("id"),
                "source_pdf": ch.get("source_pdf"),
                "page_num":   ch.get("page_num"),
                "chunk_id":   ch.get("chunk_id"),
                "start_word": ch.get("start_word"),
                "end_word":   ch.get("end_word"),
                "chunk_text": ch.get("chunk_text", ""),
            }
            for i, (score, ch) in enumerate(top_chunks)
        ],
        "prompt":           prompt,
        "generated_output": output,
        "elapsed_seconds":  elapsed,
    }

    # Save JSON
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, "task2_result.json")
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"\nResult saved → {out_path}")
    print(f"Total elapsed  : {elapsed}s  (generation: {gen_latency}s)")

    return result

## 9. Run the Pipeline

Adjust the **Configuration** cell above and run this cell.
All intermediate outputs are stored in the `result` dict.

In [15]:
# !pip install -U bitsandbytes>=0.46.1

In [15]:
result = run_pipeline()

[1/5] Loading Parquet data ...
Schema:
root
 |-- id: string (nullable = true)
 |-- source_pdf: string (nullable = true)
 |-- page_num: integer (nullable = true)
 |-- chunk_id: string (nullable = true)
 |-- start_word: integer (nullable = true)
 |-- end_word: integer (nullable = true)
 |-- chunk_text: string (nullable = true)
 |-- chunk_word_count: integer (nullable = true)
 |-- embedding: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- embedding_strategy: string (nullable = true)
 |-- embedding_dim: integer (nullable = true)

Total chunks loaded: 1872
+-----------------+--------+---------------------------+------------------+
|source_pdf       |page_num|chunk_id                   |embedding_strategy|
+-----------------+--------+---------------------------+------------------+
|6-parallel.pdf   |39      |6-parallel.pdf_p039_c000   |sbert             |
|3-stream_proc.pdf|20      |3-stream_proc.pdf_p020_c000|sbert             |
|5-nosql.pdf      |22      |5-nosq

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Result saved → /content/drive/MyDrive/DDM/distributed-retrieval-augmented-generation-team-18/task2_output/task2_result.json
Total elapsed  : 584.92s  (generation: 548.12s)


## 10. Results

In [16]:
# Retrieved chunks
print("=" * 70)
print("RETRIEVED CHUNKS")
print("=" * 70)
for ch in result["retrieved_chunks"]:
    print(f"  Rank {ch['rank']} | Score: {ch['score']:.6f} | "
          f"{ch['source_pdf']}  p.{ch['page_num']}  [{ch['chunk_id']}]")
    print(f"  {ch['chunk_text'][:250]}...")
    print()

# Prompt
print("=" * 70)
print("PROMPT SENT TO LLM")
print("=" * 70)
print(result["prompt"])

# Generated output
print("=" * 70)
print("GENERATED OUTPUT")
print("=" * 70)
print(result["generated_output"])

# Generation parameters
print("=" * 70)
print("GENERATION PARAMETERS")
print("=" * 70)
for k, v in result["generation_params"].items():
    print(f"  {k}: {v}")
print(f"  total_elapsed_seconds: {result['elapsed_seconds']}")

RETRIEVED CHUNKS
  Rank 1 | Score: 0.720810 | intro.pdf  p.2  [intro.pdf_p002_c000]
  info-h-515: second part • Second part of the course Big Data : Distributed Data Management and Scalable Analytics • Required ◦ first part of the course (throughput, latency, distributed computing, MapReduce, Lambda Architectures, Spark, Spark streami...

  Rank 2 | Score: 0.562003 | 1-intro.pdf  p.48  [1-intro.pdf_p048_c000]
  Google's solutions • New programming models and frameworks for distributed and scalable data analysis Name Purpose Google File System A distributed file system for scalable storage and high-throughput retrieval Map Reduce A programming model + execut...

  Rank 3 | Score: 0.548786 | a_ScalableMachineLearningAlgorithmsforBigDataAnalytics-ChallengesandOpportunities.pdf  p.10  [a_ScalableMachineLearningAlgorithmsforBigDataAnalytics-ChallengesandOpportunities.pdf_p010_c000]
  Journal of Artificial Intelligence Research By The Science Brigade (Publishing) Group 133 Journal of Artific

## 11. Memory Cleanup
Run between two queries to free GPU and RDD memory without restarting the kernel.

In [ ]:
import gc

def cleanup(rdd=None):
    """Unpersist cached RDD and free GPU memory (LLM model)."""
    import torch
    if rdd is not None:
        rdd.unpersist()
        print("RDD unpersisted.")
    for var_name in ["model", "pipe"]:
        if var_name in globals():
            del globals()[var_name]
    torch.cuda.empty_cache()
    gc.collect()
    allocated = torch.cuda.memory_allocated() / 1e6
    print(f"GPU memory freed. Allocated: {allocated:.1f} MB")


# Usage: call cleanup(chunk_rdd) between 2 run_pipeline()
# cleanup(chunk_rdd)

## 12. Multi-topic Generation

Run the pipeline on 4 additional topics (t03, t05, t06, t08) drawn from the evaluation set.
Each result is saved as `task2_result_<topic_id>.json` in `task2_output/`.

In [ ]:
import gc, torch, json, os

# ── Reset cached LLM to force reload with GENERATOR_MODEL (Mistral) ────────
_llm_pipeline = None
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print(f'Pipeline reset. Will load: {GENERATOR_MODEL}')

# ── Helper: save result with topic-specific filename ────────────────────────
def _save(result, topic_id):
    path = os.path.join(OUTPUT_DIR, f'task2_result_{topic_id}.json')
    with open(path, 'w') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f'Saved → {path}\n')

# ── t03 : Big Data — four Vs ─────────────────────────────────────────────────
print('=' * 60)
print('t03 — Big Data: four Vs')
print('=' * 60)
result_t03 = run_pipeline(
    query="Generate exam questions on Big Data characteristics: the four Vs — Volume, Velocity, Variety, and Veracity",
    embedding_strategy='sbert',
    similarity_strategy='cosine',
    number_of_chunks_to_retrieve=5,
    number_of_questions=3,
)
_save(result_t03, 't03')

# ── t05 : NoSQL database types ───────────────────────────────────────────────
print('=' * 60)
print('t05 — NoSQL database types')
print('=' * 60)
result_t05 = run_pipeline(
    query="Generate exam questions on NoSQL database types: key-value stores, document stores, column families, and graph databases",
    embedding_strategy='sbert',
    similarity_strategy='cosine',
    number_of_chunks_to_retrieve=5,
    number_of_questions=3,
)
_save(result_t05, 't05')

# ── t06 : CAP theorem ───────────────────────────────────────────────────────
print('=' * 60)
print('t06 — CAP theorem')
print('=' * 60)
result_t06 = run_pipeline(
    query="Generate exam questions on the CAP theorem: consistency, availability, and partition tolerance trade-offs in distributed systems",
    embedding_strategy='sbert',
    similarity_strategy='cosine',
    number_of_chunks_to_retrieve=5,
    number_of_questions=3,
)
_save(result_t06, 't06')

# ── t08 : Lambda architecture ────────────────────────────────────────────────
print('=' * 60)
print('t08 — Lambda architecture')
print('=' * 60)
result_t08 = run_pipeline(
    query="Generate exam questions on Lambda architecture for distributed stream processing: batch layer, speed layer, serving layer",
    embedding_strategy='sbert',
    similarity_strategy='cosine',
    number_of_chunks_to_retrieve=5,
    number_of_questions=3,
)
_save(result_t08, 't08')